# Static embeddings, retrofitting and semantic-space alignment

This notebook connects three questions: how distributional vectors encode similarity, how an external semantic lexicon changes geometry and how independently trained spaces can be aligned before studying semantic change.

In [2]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from nlp_portfolio.embeddings import cosine, align_orthogonal, retrofit

corpus = [
    "doctor nurse hospital patient treatment",
    "doctor hospital disease patient medicine",
    "research scientist experiment laboratory discovery",
    "scientist research laboratory data model",
    "computer algorithm data model research",
    "nurse patient care hospital treatment",
] * 80
sentences = [line.split() for line in corpus]
model = Word2Vec(sentences, vector_size=40, window=3, min_count=1,
                 workers=1, seed=42, epochs=60)
model.wv.most_similar("doctor", topn=5)

[('hospital', 0.9915186762809753),
 ('patient', 0.9891495704650879),
 ('medicine', 0.9881026744842529),
 ('disease', 0.9864515066146851),
 ('treatment', 0.9852741360664368)]

Cosine similarity depends on direction and ignores vector magnitude. The tiny corpus below is useful for checking mechanics, while linguistic conclusions require a large representative corpus and an intrinsic or downstream evaluation set.

In [4]:
terms = ["doctor", "nurse", "hospital", "scientist", "algorithm"]
similarities = pd.DataFrame(
    [[cosine(model.wv[a], model.wv[b]) for b in terms] for a in terms],
    index=terms, columns=terms,
)
similarities.round(3)

,doctor,nurse,hospital,scientist,algorithm
doctor,1.000,0.981,0.992,0.212,0.205
nurse,0.981,1.000,0.983,0.210,0.172
hospital,0.992,0.983,1.000,0.213,0.203
scientist,0.212,0.210,0.213,1.000,0.833
algorithm,0.205,0.172,0.203,0.833,1.000


## Retrofitting to a semantic lexicon

Retrofitting balances two objectives: remain close to the original distributional vector and move closer to lexicon neighbors. The anchor prevents the graph from collapsing every connected component to one point.

In [6]:
selected = {word: model.wv[word] for word in terms}
lexicon = {
    "doctor": ["nurse", "hospital"],
    "nurse": ["doctor", "hospital"],
    "hospital": ["doctor", "nurse"],
    "scientist": [], "algorithm": [],
}
updated = retrofit(selected, lexicon, iterations=10, anchor_weight=1.0)
pd.DataFrame({
    "pair": ["doctor–nurse", "doctor–scientist"],
    "before": [cosine(selected["doctor"], selected["nurse"]),
               cosine(selected["doctor"], selected["scientist"])],
    "after": [cosine(updated["doctor"], updated["nurse"]),
              cosine(updated["doctor"], updated["scientist"])],
}).round(3)

,pair,before,after
0,doctor–nurse,0.981,0.999
1,doctor–scientist,0.212,0.213


## Orthogonal alignment

Word2Vec spaces are identifiable only up to rotation. Directly subtracting vectors from two independently trained models is therefore invalid. Orthogonal Procrustes estimates the rotation from shared anchor words while preserving inner products.

In [8]:
rng = np.random.default_rng(42)
source = rng.normal(size=(60, 20))
true_rotation, _ = np.linalg.qr(rng.normal(size=(20, 20)))
target = source @ true_rotation + rng.normal(scale=0.01, size=source.shape)
estimated = align_orthogonal(source[:40], target[:40])
before_mse = np.mean((source[40:] - target[40:]) ** 2)
after_mse = np.mean((source[40:] @ estimated - target[40:]) ** 2)
print(f"held-out MSE before alignment={before_mse:.4f}")
print(f"held-out MSE after alignment={after_mse:.4f}")

held-out MSE before alignment=2.1442
held-out MSE after alignment=0.0001


## Optional comparison with pretrained vectors

The cell downloads GloVe through Gensim. It is disabled by default because the checkpoint is large. The evaluation avoids querying nearest neighbors by hand and instead uses a fixed list of similarity pairs.

In [10]:
RUN_PRETRAINED = False
if RUN_PRETRAINED:
    import gensim.downloader as api
    glove = api.load("glove-wiki-gigaword-100")
    evaluation_pairs = [("doctor", "nurse"), ("doctor", "banana"), ("virus", "disease")]
    display(pd.DataFrame([
        {"word_1": a, "word_2": b, "cosine": glove.similarity(a, b)}
        for a, b in evaluation_pairs
    ]))

### Interpretation limits

Nearest neighbors reflect corpus frequency, genre and social bias. Retrofitting imports the assumptions of its lexicon. Diachronic comparisons require matched preprocessing, stable anchors and uncertainty estimates over corpus resamples.